In [32]:
# ========================================
# Test Dataset Loader
# ========================================

import sys
sys.path.append('../')

from src.dataset import NTIRETrainDataset, get_transforms
from torch.utils.data import DataLoader

# Get transforms
train_transform, val_transform = get_transforms()

# ========================================
# Test 1: Load single shard
# ========================================
print("=" * 60)
print("TEST 1: Single shard loading")
print("=" * 60)

dataset = NTIRETrainDataset(
    shard_dir='../data/train',
    shard_nums=[0],  # Only first shard for quick test
    transform=train_transform
)

# Check first sample
image, label = dataset[0]
print(f"\nFirst sample:")
print(f"  Image shape: {image.shape}")
print(f"  Image dtype: {image.dtype}")
print(f"  Label: {label} ({'Real' if label == 0 else 'Fake'})")




# ========================================
# Test 2: DataLoader batch
# ========================================
print("\n" + "=" * 60)
print("TEST 2: DataLoader batch")
print("=" * 60)

loader = DataLoader(dataset, batch_size=8, shuffle=True, num_workers=0)
batch_images, batch_labels = next(iter(loader))

print(f"\nBatch shape: {batch_images.shape}")
print(f"Batch labels: {batch_labels.tolist()}")
print(f"Label distribution: {sum(batch_labels).item()} fake, {len(batch_labels) - sum(batch_labels).item()} real")

# ========================================
# Test 3: Load all shards
# ========================================
print("\n" + "=" * 60)
print("TEST 3: All shards loading")
print("=" * 60)

full_dataset = NTIRETrainDataset(
    shard_dir='../data/train',
    shard_nums=None,  # All shards
    transform=train_transform
)

print("\n✅ Dataset loader working correctly!")


TEST 1: Single shard loading
Loaded 1 shards: 50000 images
  Real: 17982 | Fake: 32018

First sample:
  Image shape: torch.Size([3, 448, 448])
  Image dtype: torch.float32
  Label: 0 (Real)

TEST 2: DataLoader batch

Batch shape: torch.Size([8, 3, 448, 448])
Batch labels: [0, 1, 1, 1, 0, 0, 1, 0]
Label distribution: 4 fake, 4 real

TEST 3: All shards loading
Loaded 6 shards: 277643 images
  Real: 100000 | Fake: 177643

✅ Dataset loader working correctly!


In [ ]:
# ========================================
# Analyze image dimensions
# ========================================
print("\n" + "=" * 60)
print("ANALYSIS: Image dimensions")
print("=" * 60)

from PIL import Image
import numpy as np
import os

# Sample images from dataset
sizes = []
for idx in range(min(5000, len(dataset))):  # Sample 5k images
    row = dataset.label_df.iloc[idx]
    img_path = os.path.join(row['shard_path'], 'images', row['image_name'])
    img = Image.open(img_path)
    sizes.append((img.width, img.height))

widths = [s[0] for s in sizes]
heights = [s[1] for s in sizes]
min_dims = [min(w, h) for w, h in sizes]

print(f"\nWidth  - Min: {min(widths)}, Max: {max(widths)}, Mean: {np.mean(widths):.0f}")
print(f"Height - Min: {min(heights)}, Max: {max(heights)}, Mean: {np.mean(heights):.0f}")
print(f"Min dimension - Min: {min(min_dims)}, Max: {max(min_dims)}")

# Find safe crop size (99th percentile of min dimension)
safe_crop_size = int(np.percentile(min_dims, 1))  # 99% of images can be cropped
print(f"\nSuggested crop size (99% coverage): {safe_crop_size}")